In [17]:
import numpy as np
import pandas as pd
from scipy.stats import t

# 1. Configuração do Cenário (Selic 15% a.a.)
taxa_livre_risco_anual = 0.15

# 2. Definição da Função com Distribuição t e Monitoramento
def calculate_sharpe_ratio_robust(portfolio_data, portfolio_returns_dict, risk_free_rate):
    sharpe_results = {}

    # Tenta capturar o dicionário 'retornos' do ambiente global
    dados_historicos = globals().get('retornos', None)

    for nome, portfolio in portfolio_data.items():
        portfolio_returns_list = []
        ativos_encontrados = []
        ativos_faltantes = []

        if dados_historicos is not None:
            for acao in portfolio:
                if acao in dados_historicos and not dados_historicos[acao].empty:
                    portfolio_returns_list.append(dados_historicos[acao])
                    ativos_encontrados.append(acao)
                else:
                    ativos_faltantes.append(acao)

        # Log de monitoramento para o usuário
        if ativos_faltantes:
            print(f"DEBUG [{nome}]: Calculando sem os ativos (dados ausentes): {ativos_faltantes}")

        if portfolio_returns_list:
            # Concatena os retornos para cálculo da volatilidade do portfólio
            portfolio_df = pd.concat(portfolio_returns_list, axis=1, join='outer')
            portfolio_monthly_returns = portfolio_df.mean(axis=1).dropna()

            if len(portfolio_monthly_returns) > 1:
                # --- AJUSTE ROBUSTO (DISTRIBUIÇÃO T) ---
                # t.fit estima a 'escala', que é a volatilidade robusta
                data_points = portfolio_monthly_returns.values.astype(float)
                _, _, scale_mensal = t.fit(data_points)
                volatilidade_anualizada = scale_mensal * np.sqrt(12)
            else:
                volatilidade_anualizada = 0
        else:
            # Fallback: Se não houver nenhum dado histórico no grupo
            vols_estimadas = {'Risco Baixo': 0.12, 'Risco Médio': 0.18, 'Risco Alto': 0.30}
            volatilidade_anualizada = vols_estimadas.get(nome, 0.20)

        # Cálculo do Sharpe Ratio
        retorno_anualizado = portfolio_returns_dict[nome] * 12
        if volatilidade_anualizada > 0:
            sharpe_results[nome] = (retorno_anualizado - risk_free_rate) / volatilidade_anualizada
        else:
            sharpe_results[nome] = np.nan

    return sharpe_results

# 3. Execução dos Cálculos
print("--- Verificação de Dados Históricos ---")
sharpe_ratios = calculate_sharpe_ratio_robust(portfolios, retornos_portfolios, taxa_livre_risco_anual)
sharpe_ratios_diversificados = calculate_sharpe_ratio_robust(portfolios_diversificados, retornos_portfolios_diversificados, taxa_livre_risco_anual)
print("-" * 40)

# 4. Impressão formatada idêntica ao seu notebook
print("\nPortfólios:")
for nome, portfolio in portfolios.items():
    print(f"{nome}: {portfolio}")

print("\nPortfólios Diversificados:")
for nome, portfolio in portfolios_diversificados.items():
    print(f"{nome}: {portfolio}")

print("\nRetornos Mensais dos Portfólios:")
for nome, retorno in retornos_portfolios.items():
    print(f"{nome}: {retorno:.2%}" if not np.isnan(retorno) else f"{nome}: N/A")

print("\nRetornos Mensais dos Portfólios Diversificados:")
for nome, retorno in retornos_portfolios_diversificados.items():
    print(f"{nome}: {retorno:.2%}" if not np.isnan(retorno) else f"{nome}: N/A")

print("\nSharpe Ratios (Portfólios Originais):")
for nome, ratio in sharpe_ratios.items():
    if not np.isnan(ratio):
        print(f"{nome}: {ratio:.2f}")
    else:
        print(f"{nome}: N/A (dados insuficientes ou volatilidade zero)")

print("\nSharpe Ratios (Portfólios Diversificados):")
for nome, ratio in sharpe_ratios_diversificados.items():
    if not np.isnan(ratio):
        print(f"{nome}: {ratio:.2f}")
    else:
        print(f"{nome}: N/A (dados insuficientes ou volatilidade zero)")

--- Verificação de Dados Históricos ---
----------------------------------------

Portfólios:
Risco Baixo: ['ITUB4.SA', 'ABEV3.SA', 'BBDC4.SA', 'ITSA4.SA', 'KLBN11.SA', 'RADL3.SA', 'SUZB3.SA']
Risco Médio: ['VALE3.SA', 'BBAS3.SA', 'B3SA3.SA', 'LREN3.SA', 'MRVE3.SA', 'RENT3.SA']
Risco Alto: ['PETR4.SA', 'ENEV3.SA', 'BPAC11.SA', 'CSNA3.SA', 'GGBR4.SA', 'MGLU3.SA', 'PCAR3.SA']

Portfólios Diversificados:
Risco Baixo: ['ITUB4.SA', 'ABEV3.SA', 'KLBN11.SA', 'RADL3.SA']
Risco Médio: ['VALE3.SA', 'BBAS3.SA', 'B3SA3.SA', 'LREN3.SA', 'MRVE3.SA']
Risco Alto: ['PETR4.SA', 'BPAC11.SA', 'ENEV3.SA', 'CSNA3.SA', 'MGLU3.SA']

Retornos Mensais dos Portfólios:
Risco Baixo: 1.18%
Risco Médio: 1.36%
Risco Alto: 1.91%

Retornos Mensais dos Portfólios Diversificados:
Risco Baixo: 1.31%
Risco Médio: 1.21%
Risco Alto: 1.87%

Sharpe Ratios (Portfólios Originais):
Risco Baixo: -0.07
Risco Médio: 0.07
Risco Alto: 0.26

Sharpe Ratios (Portfólios Diversificados):
Risco Baixo: 0.06
Risco Médio: -0.03
Risco Alto: 0.2